<a href="https://colab.research.google.com/github/Dedeepya0510/CSA4703---Deep-Learning/blob/main/Lab%20Experiment/Exp_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q keras-tuner

import tensorflow as tf
import keras_tuner as kt
import pandas as pd
import time
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load small IMDB subset
vocab_size = 5000
max_length = 100

(X_train, y_train), (X_test, y_test) = imdb.load_data(
    num_words=vocab_size
)

X_train = pad_sequences(X_train[:2000], maxlen=max_length)
y_train = y_train[:2000]

X_test = pad_sequences(X_test[:500], maxlen=max_length)
y_test = y_test[:500]

def build_model(hp, cell):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Input(shape=(max_length,)))

    emb = hp.Choice('embedding', [16, 32])
    units = hp.Choice('units', [16, 32])
    dropout = hp.Choice('dropout', [0.2, 0.4])

    model.add(tf.keras.layers.Embedding(vocab_size, emb))

    if cell == 'RNN':
        model.add(tf.keras.layers.SimpleRNN(units))
    elif cell == 'LSTM':
        model.add(tf.keras.layers.LSTM(units))
    else:
        model.add(tf.keras.layers.GRU(units))

    model.add(tf.keras.layers.Dropout(dropout))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))

    lr = hp.Choice('learning_rate', [0.001, 0.0005])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

results = []

for cell in ['RNN', 'LSTM', 'GRU']:
    tuner = kt.RandomSearch(
        hypermodel=lambda hp: build_model(hp, cell),
        objective='val_accuracy',
        max_trials=2,
        directory='tuning',
        project_name=cell,
        overwrite=True
    )

    start = time.time()

    tuner.search(
        X_train, y_train,
        epochs=2,
        batch_size=64,
        validation_split=0.2,
        verbose=0
    )

    elapsed = time.time() - start

    best = tuner.get_best_models(1)[0]
    hp = tuner.get_best_hyperparameters(1)[0]

    loss, accuracy = best.evaluate(X_test, y_test, verbose=0)

    results.append([
        cell, accuracy, best.count_params(), round(elapsed, 2),
        hp.get('units'), hp.get('embedding'),
        hp.get('dropout'), hp.get('learning_rate')
    ])

df = pd.DataFrame(results, columns=[
    'Model', 'Accuracy', 'Parameters', 'Training Time',
    'Units', 'Embedding', 'Dropout', 'Learning Rate'
])

print(df.to_string(index=False))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.13/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


Model  Accuracy  Parameters  Training Time  Units  Embedding  Dropout  Learning Rate
  RNN     0.508      162113          10.46     32         32      0.4          0.001
 LSTM     0.686      163153          10.24     16         32      0.2          0.001
  GRU     0.532       81649          13.29     16         16      0.2          0.001
